(C) Model Prediction Explanation Pipeline: 

In [1]:
import os
import re
import json
import joblib
import requests
import jsonschema
import pandas as pd
from dotenv import load_dotenv

1. LLM API connection

In [2]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=500):
    """
    Calls the LLM API with structured prompts.
    Requires environment variable LLM_API_KEY.
    """
    api_key = os.getenv("OPENROUTER_APIKEY")  
    if not api_key:
        raise ValueError("Missing LLM_API_KEY environment variable")

    url = "https://openrouter.ai/api/v1/chat/completions"
    payload = {
        "model": "openai/gpt-4o-mini", 
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        print(f"LLM API error: {response.status_code}")
        return None

    return response.json()["choices"][0]["message"]["content"]

#  Demo

print(call_llm("You are a bot", "Reply with only the word: hello"))


ConnectionError: HTTPSConnectionPool(host='openrouter.ai', port=443): Max retries exceeded with url: /api/v1/chat/completions (Caused by NameResolutionError("HTTPSConnection(host='openrouter.ai', port=443): Failed to resolve 'openrouter.ai' ([Errno 11001] getaddrinfo failed)"))

2.Prompts

In [ ]:
system_prompt = """
You are a structured data extractor.
You must output ONLY valid JSON that matches this schema:

{
  "prediction_label": "string",
  "confidence_level": "low|medium|high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}

Do not include any text outside the JSON.
Do not add explanations, comments, or formatting.
"""

user_prompt_template = """
Features: {features}
Prediction: {pred_class}
Probability: {pred_proba:.3f}
"""




3. Guardrail for PII

In [ ]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# ✅ Demo
print("Email test:", has_pii("Contact me at test@example.com"))  # True
print("Safe test:", has_pii("This is a safe input"))             # False


Email test: True
Safe test: False


4. Schema Definition

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"]
}


5. Helpers

In [ ]:
def make_input_template(feature_names, overrides=None):
    template = {name: 0 for name in feature_names}
    if overrides:
        template.update(overrides)
    return pd.DataFrame([template])

def encode_record(df_row):
    if len(df_row) != 1:
        raise ValueError("encode_record expects a single-row DataFrame")
    return df_row.values

6. Model Load + Feature Names

In [ ]:
model = joblib.load("best_model.pkl")
feature_names = model.feature_names_in_ 

7. Pipeline Runner

In [ ]:
def run_pipeline(inputs, temp=0.0):
    for features in inputs:
        encoded = encode_record(features)
        pred_class = model.predict(encoded)[0]
        pred_proba = model.predict_proba(encoded).max()

        # Convert DataFrame row to dict before JSON serialization
        user_prompt = user_prompt_template.format(
            features=json.dumps(features.to_dict(orient="records")[0]),
            pred_class=pred_class,
            pred_proba=pred_proba
        )

        if has_pii(user_prompt):
            print("Input blocked: PII detected.")
            continue

        raw_response = call_llm(system_prompt, user_prompt, temperature=temp)
        if raw_response is None:
            print("LLM call failed. Skipping this input.")
            continue

        try:
            parsed = json.loads(raw_response.strip())
            jsonschema.validate(instance=parsed, schema=schema)
            valid = True
        except (json.JSONDecodeError, jsonschema.ValidationError) as e:
            print(f"Validation error: {e}")
            parsed = {k: None for k in schema["required"]}
            valid = False

        print("Features:", features.to_dict(orient="records")[0])
        print("Predicted Class:", pred_class)
        print("Probability:", pred_proba)
        print("Raw LLM Response:", raw_response)
        print("LLM Explanation:", parsed)
        print("Validation Status:", "PASS" if valid else "FAIL")

8. Demo Run

In [ ]:
if __name__ == "__main__":
    test_inputs = [
        make_input_template(feature_names, {"State_Bihar":1,"City_Patna":1,"Furnished_Status_Semi-furnished":1,"BHK":3,"Size_in_SqFt":1200}),
        make_input_template(feature_names, {"State_Delhi":1,"City_New Delhi":1,"Property_Type_Villa":1,"BHK":4,"Size_in_SqFt":2000}),
        make_input_template(feature_names, {"State_Maharashtra":1,"City_Mumbai":1,"Furnished_Status_Unfurnished":1,"BHK":2,"Size_in_SqFt":800})
    ]
    run_pipeline(test_inputs, temp=0.0)

c:\Users\a\OneDrive\Desktop\indian housing prices\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
c:\Users\a\OneDrive\Desktop\indian housing prices\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


Features: {'BHK': 3, 'Size_in_SqFt': 1200, 'Floor_No': 0, 'Total_Floors': 0, 'Age_of_Property': 0, 'Nearby_Schools': 0, 'Nearby_Hospitals': 0, 'Public_Transport_Accessibility': 0, 'Parking_Space': 0, 'Security': 0, 'State_Assam': 0, 'State_Bihar': 1, 'State_Chhattisgarh': 0, 'State_Delhi': 0, 'State_Gujarat': 0, 'State_Haryana': 0, 'State_Jharkhand': 0, 'State_Karnataka': 0, 'State_Kerala': 0, 'State_Madhya Pradesh': 0, 'State_Maharashtra': 0, 'State_Odisha': 0, 'State_Punjab': 0, 'State_Rajasthan': 0, 'State_Tamil Nadu': 0, 'State_Telangana': 0, 'State_Uttar Pradesh': 0, 'State_Uttarakhand': 0, 'State_West Bengal': 0, 'City_Amritsar': 0, 'City_Bangalore': 0, 'City_Bhopal': 0, 'City_Bhubaneswar': 0, 'City_Bilaspur': 0, 'City_Chennai': 0, 'City_Coimbatore': 0, 'City_Cuttack': 0, 'City_Dehradun': 0, 'City_Durgapur': 0, 'City_Dwarka': 0, 'City_Faridabad': 0, 'City_Gaya': 0, 'City_Gurgaon': 0, 'City_Guwahati': 0, 'City_Haridwar': 0, 'City_Hyderabad': 0, 'City_Indore': 0, 'City_Jaipur': 0, 

c:\Users\a\OneDrive\Desktop\indian housing prices\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
c:\Users\a\OneDrive\Desktop\indian housing prices\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


Features: {'BHK': 4, 'Size_in_SqFt': 2000, 'Floor_No': 0, 'Total_Floors': 0, 'Age_of_Property': 0, 'Nearby_Schools': 0, 'Nearby_Hospitals': 0, 'Public_Transport_Accessibility': 0, 'Parking_Space': 0, 'Security': 0, 'State_Assam': 0, 'State_Bihar': 0, 'State_Chhattisgarh': 0, 'State_Delhi': 1, 'State_Gujarat': 0, 'State_Haryana': 0, 'State_Jharkhand': 0, 'State_Karnataka': 0, 'State_Kerala': 0, 'State_Madhya Pradesh': 0, 'State_Maharashtra': 0, 'State_Odisha': 0, 'State_Punjab': 0, 'State_Rajasthan': 0, 'State_Tamil Nadu': 0, 'State_Telangana': 0, 'State_Uttar Pradesh': 0, 'State_Uttarakhand': 0, 'State_West Bengal': 0, 'City_Amritsar': 0, 'City_Bangalore': 0, 'City_Bhopal': 0, 'City_Bhubaneswar': 0, 'City_Bilaspur': 0, 'City_Chennai': 0, 'City_Coimbatore': 0, 'City_Cuttack': 0, 'City_Dehradun': 0, 'City_Durgapur': 0, 'City_Dwarka': 0, 'City_Faridabad': 0, 'City_Gaya': 0, 'City_Gurgaon': 0, 'City_Guwahati': 0, 'City_Haridwar': 0, 'City_Hyderabad': 0, 'City_Indore': 0, 'City_Jaipur': 0, 

c:\Users\a\OneDrive\Desktop\indian housing prices\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
c:\Users\a\OneDrive\Desktop\indian housing prices\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


Features: {'BHK': 2, 'Size_in_SqFt': 800, 'Floor_No': 0, 'Total_Floors': 0, 'Age_of_Property': 0, 'Nearby_Schools': 0, 'Nearby_Hospitals': 0, 'Public_Transport_Accessibility': 0, 'Parking_Space': 0, 'Security': 0, 'State_Assam': 0, 'State_Bihar': 0, 'State_Chhattisgarh': 0, 'State_Delhi': 0, 'State_Gujarat': 0, 'State_Haryana': 0, 'State_Jharkhand': 0, 'State_Karnataka': 0, 'State_Kerala': 0, 'State_Madhya Pradesh': 0, 'State_Maharashtra': 1, 'State_Odisha': 0, 'State_Punjab': 0, 'State_Rajasthan': 0, 'State_Tamil Nadu': 0, 'State_Telangana': 0, 'State_Uttar Pradesh': 0, 'State_Uttarakhand': 0, 'State_West Bengal': 0, 'City_Amritsar': 0, 'City_Bangalore': 0, 'City_Bhopal': 0, 'City_Bhubaneswar': 0, 'City_Bilaspur': 0, 'City_Chennai': 0, 'City_Coimbatore': 0, 'City_Cuttack': 0, 'City_Dehradun': 0, 'City_Durgapur': 0, 'City_Dwarka': 0, 'City_Faridabad': 0, 'City_Gaya': 0, 'City_Gurgaon': 0, 'City_Guwahati': 0, 'City_Haridwar': 0, 'City_Hyderabad': 0, 'City_Indore': 0, 'City_Jaipur': 0, '